In [ ]:
!pip install pandas numpy faiss-cpu openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 98.9 MB/s eta 0:00:00


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define Folder Path
DRIVE = "/content/drive"
FOLDER_DATA = r"/content/drive/My Drive/Thesis/CUI Mapping/Data/"

# Define the dataset filename
DATA_FILE_NAME_UMLS = "UMLS_ENG_CUI_STR_TTY_STY.csv"
DATA_FILE_NAME_THREE_SYMPTOMS = "all three symptoms-156-sorted-20241112.txt"
DATA_FILE_NAME_FAISS = "UMLS_ENG_CUI_STR_TTY_STY_FAISS.index"
DATA_FILE_NAME_LOOKUP = "UMLS_ENG_CUI_STR_TTY_STY_LOOKUP.pkl"

# Full Path to Data File
DATA_FILE_PATH_UMLS = FOLDER_DATA + DATA_FILE_NAME_UMLS
DATA_FILE_PATH_THREE_SYMPTOMS = FOLDER_DATA + DATA_FILE_NAME_THREE_SYMPTOMS
DATA_FILE_PATH_FAISS = FOLDER_DATA + DATA_FILE_NAME_FAISS
DATA_FILE_PATH_LOOKUP = FOLDER_DATA + DATA_FILE_NAME_LOOKUP

Mounted at /content/drive


In [ ]:
import torch
print("CUDA available?", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA available? True
GPU name: NVIDIA A100-SXM4-40GB


In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [ ]:
import pandas as pd

# Load your preprocessed UMLS data
df_final = pd.read_csv(DATA_FILE_PATH_UMLS)

# Lists for FAISS/SapBERT
umls_terms = df_final["STR"].tolist()
umls_cuis  = df_final["CUI"].tolist()
umls_tty  = df_final["TTY"].tolist()
umls_sty  = df_final["STY"].tolist()

In [ ]:
import numpy as np
import time

def encode_batch(texts, batch_size, max_length, progress_every=100000):
    start_time = time.time()
    all_vecs = []
    next_progress = progress_every

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                        max_length=max_length).to(device)
        with torch.no_grad():
            out = model(**enc)
        pooled = out.last_hidden_state[:,0,:]
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_vecs.append(pooled.cpu().numpy().astype("float32"))

        processed = i + len(batch)

        if processed >= next_progress:
            print(f"Processed ~{next_progress:,} / {len(texts):,} terms")
            next_progress += progress_every

    embeddings = np.vstack(all_vecs)
    total_time = time.time() - start_time
    print(f"\nFinished processing {len(texts):,} terms in {total_time/60:.2f} minutes")
    return embeddings

In [ ]:
umls_embeddings = encode_batch(umls_terms, batch_size=128, max_length=32)
print("Embeddings shape:", umls_embeddings.shape)

Processed ~100,000 / 3,594,105 terms
Processed ~200,000 / 3,594,105 terms
Processed ~300,000 / 3,594,105 terms
Processed ~400,000 / 3,594,105 terms
Processed ~500,000 / 3,594,105 terms
Processed ~600,000 / 3,594,105 terms
Processed ~700,000 / 3,594,105 terms
Processed ~800,000 / 3,594,105 terms
Processed ~900,000 / 3,594,105 terms
Processed ~1,000,000 / 3,594,105 terms
Processed ~1,100,000 / 3,594,105 terms
Processed ~1,200,000 / 3,594,105 terms
Processed ~1,300,000 / 3,594,105 terms
Processed ~1,400,000 / 3,594,105 terms
Processed ~1,500,000 / 3,594,105 terms
Processed ~1,600,000 / 3,594,105 terms
Processed ~1,700,000 / 3,594,105 terms
Processed ~1,800,000 / 3,594,105 terms
Processed ~1,900,000 / 3,594,105 terms
Processed ~2,000,000 / 3,594,105 terms
Processed ~2,100,000 / 3,594,105 terms
Processed ~2,200,000 / 3,594,105 terms
Processed ~2,300,000 / 3,594,105 terms
Processed ~2,400,000 / 3,594,105 terms
Processed ~2,500,000 / 3,594,105 terms
Processed ~2,600,000 / 3,594,105 terms
Proc

In [ ]:
import faiss

index = faiss.IndexFlatIP(umls_embeddings.shape[1])  # cosine similarity
index.add(umls_embeddings)
print("FAISS index built with", index.ntotal, "vectors")

FAISS index built with 3594105 vectors


In [ ]:
import faiss
import pickle

# Save FAISS index
faiss.write_index(index, DATA_FILE_PATH_FAISS)

# Save lookup
lookup_dict = {
    "cuis": df_final["CUI"].tolist(),
    "terms": df_final["STR"].tolist(),
    "ttys": df_final["TTY"].tolist(),
    "stys": df_final["STY"].tolist()
}

with open(DATA_FILE_PATH_LOOKUP, "wb") as f:
    pickle.dump(lookup_dict, f)

print("Saved FAISS index and lookup tables")

Saved FAISS index and lookup tables
